# PyGeoFetch — 05: Pipelines & Scheduling

This notebook covers the pipeline orchestration system: YAML-defined workflows, step types, scheduling with cron expressions, and running pipelines from Python.

---
### What you'll learn
- Write YAML pipelines with search, filter, download, and export steps
- Run pipelines programmatically from Python
- Schedule pipelines with cron expressions
- Build a weekly monitoring workflow
- Inspect execution history

In [ ]:
from pygeofetch import PyGeoFetch
from pygeofetch.core.scheduler import PipelineScheduler, Pipeline, PipelineStep
from pathlib import Path
import yaml

client = PyGeoFetch(log_level="INFO")
scheduler = PipelineScheduler(engine=client)
Path("output/pipelines").mkdir(parents=True, exist_ok=True)
print("Ready")

## 1. Pipeline Structure

In [ ]:
# A pipeline has: name, optional schedule (cron), optional description, and steps
# Each step is a dict with exactly one key (the step type)

# Available step types:
step_types = {
    "search":   "Query providers with bbox, date range, cloud filter",
    "filter":   "Apply a Python expression to narrow search results",
    "download": "Download scenes with parallel workers and bands",
    "process":  "Apply processing actions (stub — add custom processors)",
    "export":   "Send results to destination (S3, local path)",
}

print("Pipeline step types:")
for step, desc in step_types.items():
    print(f"  {step:<12} — {desc}")

## 2. Simple Pipeline — Search and Download

In [ ]:
# Write a simple pipeline YAML
simple_pipeline_yaml = """
name: nyc-sentinel2-rgb
description: Download clear Sentinel-2 RGB scenes over New York City

steps:
  - search:
      providers: [aws_earth, planetary_computer]
      bbox: "-74.1,40.6,-73.7,40.9"
      start_date: "2024-01-01"
      end_date: "2024-06-01"
      cloud_cover: "0-10"
      max_results: 20

  - filter:
      expression: "data.cloud_cover < 5"

  - download:
      parallel: 2
      output: ./output/pipelines/rgb/
      verify_checksum: false
"""

simple_path = Path("output/pipelines/simple.yaml")
simple_path.write_text(simple_pipeline_yaml)
print(f"Pipeline written → {simple_path}")
print(yaml.safe_load(simple_pipeline_yaml))

In [ ]:
# Validate the pipeline without running it
pipeline_obj = scheduler.load_pipeline(simple_path)
print(f"Pipeline: {pipeline_obj.name!r}")
print(f"Steps:    {len(pipeline_obj.steps)}")
print(f"Schedule: {pipeline_obj.schedule or 'none (run manually)'}")
print()
for i, step in enumerate(pipeline_obj.steps, 1):
    print(f"  Step {i}: {step.step_type}")
    for k, v in step.config.items():
        print(f"    {k}: {v}")

In [ ]:
# Run the pipeline
print(f"Running pipeline: {pipeline_obj.name!r}...")
result = scheduler.run_once(pipeline_obj.name)

print(f"\nResult:")
print(f"  Pipeline: {result['pipeline']}")
print(f"  Duration: {result['duration_seconds']:.1f}s")
print(f"  Success:  {result['success']}")
print()
print("Steps:")
for step in result["steps"]:
    status_icon = "✓" if step["status"] == "ok" else "✗"
    print(f"  {status_icon} {step['step']}: {step['status']}")
    if step.get("error"):
        print(f"    Error: {step['error']}")

## 3. Full Pipeline with All Steps

In [ ]:
full_pipeline_yaml = """
name: weekly-monitoring
schedule: "0 6 * * 1"  # Every Monday at 06:00 UTC
description: |
  Weekly Sentinel-2 acquisition for change monitoring.
  Searches 3 free providers, filters to best quality,
  downloads NDVI bands, and exports results.

steps:
  - search:
      providers: [aws_earth, planetary_computer, element84]
      bbox: "-74.1,40.6,-73.7,40.9"
      date_range: last_7_days
      cloud_cover: "0-15"
      max_results: 30

  - filter:
      expression: "data.cloud_cover < 10"

  - download:
      parallel: 3
      output: ./output/pipelines/weekly/
      verify_checksum: false

  - export:
      format: cloud_optimized_geotiff
      destination: ./output/pipelines/weekly_export/
"""

full_path = Path("output/pipelines/weekly.yaml")
full_path.write_text(full_pipeline_yaml)
print(f"Full pipeline written → {full_path}")

In [ ]:
# Load and inspect
weekly = scheduler.load_pipeline(full_path)
print(f"Pipeline: {weekly.name!r}")
print(f"Schedule: {weekly.schedule!r} (Every Monday 06:00 UTC)")
print(f"Steps: {len(weekly.steps)}")

# Show next scheduled run
try:
    import croniter
    from datetime import datetime
    itr = croniter.croniter(weekly.schedule, datetime.utcnow())
    next_run = itr.get_next(datetime)
    print(f"Next run: {next_run.strftime('%Y-%m-%d %H:%M UTC')}")
except ImportError:
    print("Install croniter for cron scheduling: pip install croniter")

## 4. Build Pipelines Programmatically (Python API)

In [ ]:
# Build a pipeline without YAML
pipeline_from_python = Pipeline(
    name="python-api-pipeline",
    description="Built programmatically in Python",
    schedule="0 8 * * *",  # Daily at 08:00 UTC
    steps=[
        PipelineStep(
            step_type="search",
            config={
                "providers": ["aws_earth"],
                "bbox": "-74.1,40.6,-73.7,40.9",
                "date_range": "last_7_days",
                "cloud_cover": "0-20",
                "max_results": 10,
            },
        ),
        PipelineStep(
            step_type="filter",
            config={"expression": "data.cloud_cover < 15"},
        ),
        PipelineStep(
            step_type="download",
            config={
                "parallel": 2,
                "output": "./output/pipelines/daily/",
                "verify_checksum": False,
            },
        ),
    ],
)

scheduler.add_pipeline(pipeline_from_python)
print(f"Added pipeline: {pipeline_from_python.name!r}")
print(f"Steps: {[s.step_type for s in pipeline_from_python.steps]}")

# Export to YAML
pipeline_dict = {
    "name": pipeline_from_python.name,
    "schedule": pipeline_from_python.schedule,
    "description": pipeline_from_python.description,
    "steps": [{s.step_type: s.config} for s in pipeline_from_python.steps],
}
export_path = Path("output/pipelines/python_api_pipeline.yaml")
with open(export_path, "w") as f:
    yaml.dump(pipeline_dict, f, default_flow_style=False)
print(f"\nExported to YAML → {export_path}")
print(open(export_path).read())

## 5. Pipeline Management

In [ ]:
# List all registered pipelines
pipelines = scheduler.list_pipelines()
print(f"Registered pipelines ({len(pipelines)}):")
print(f"{'Name':<35} {'Steps':<8} {'Schedule':<25} {'Last Run':<20} {'Runs'}")
print("-" * 100)
for p in pipelines:
    last = p["last_run"][:16] if p["last_run"] else "never"
    sched = p["schedule"] or "manual only"
    print(f"{p['name']:<35} {p['steps']:<8} {sched:<25} {last:<20} {p['run_count']}")

In [ ]:
# Run a specific pipeline step-by-step
print("Running weekly-monitoring pipeline...")
result = scheduler.run_once("weekly-monitoring")

import json
print(json.dumps({
    "pipeline": result["pipeline"],
    "duration_seconds": round(result["duration_seconds"], 2),
    "success": result["success"],
    "steps": [
        {"step": s["step"], "status": s["status"]}
        for s in result["steps"]
    ]
}, indent=2))

## 6. Cron Schedule Reference

In [ ]:
# Common cron expressions for satellite monitoring
schedules = [
    ("0 6 * * 1",     "Every Monday at 06:00 UTC"),
    ("0 6 * * *",     "Every day at 06:00 UTC"),
    ("0 */6 * * *",   "Every 6 hours"),
    ("0 6 1 * *",     "First of every month at 06:00"),
    ("0 6 * * 1,4",   "Monday and Thursday at 06:00"),
    ("30 5 * * 2-6",  "Weekdays at 05:30 UTC"),
    ("0 0 1 1 *",     "Once a year (Jan 1st)"),
]

print("Common cron schedules for satellite pipelines:")
print(f"{'Cron Expression':<22} {'Description':<40} {'Next Run'}")
print("-" * 80)

try:
    import croniter
    from datetime import datetime
    now = datetime.utcnow()
    for cron, desc in schedules:
        itr = croniter.croniter(cron, now)
        nxt = itr.get_next(datetime).strftime("%Y-%m-%d %H:%M")
        print(f"  {cron:<20} {desc:<40} {nxt}")
except ImportError:
    for cron, desc in schedules:
        print(f"  {cron:<20} {desc}")
    print("\n  Install croniter for next-run calculation: pip install croniter")

In [ ]:
# CLI equivalents for pipeline management
print("CLI equivalents:")
print()
print("# Validate")
print("PyGeoFetch pipeline validate output/pipelines/weekly.yaml")
print()
print("# Run once")
print("PyGeoFetch pipeline run output/pipelines/weekly.yaml")
print()
print("# Schedule")
print('PyGeoFetch pipeline schedule output/pipelines/weekly.yaml --name "weekly-monitoring"')
print()
print("# List scheduled")
print("PyGeoFetch pipeline list-scheduled")
print()
print("# View history")
print("PyGeoFetch pipeline history --limit 10")
print()
print("# Unschedule")
print("PyGeoFetch pipeline unschedule weekly-monitoring")

---
## Summary
- ✅ Wrote YAML pipelines with search, filter, download, and export steps
- ✅ Ran pipelines programmatically and via the scheduler
- ✅ Built pipelines in Python without YAML
- ✅ Scheduled with cron expressions
- ✅ Inspected execution history and listing

**Next:** `06_real_world_workflows.ipynb` — end-to-end workflows: NDVI time series, change detection, multi-sensor fusion, SAR analysis.